# 🎵 NLP 기초: 텍스트 분석과 검색 엔진 만들기

## 📚 학습 목표
- 텍스트 데이터의 특성 이해
- 자연어 처리(NLP) 기본 개념 학습
- BOW(Bag of Words)와 TF-IDF 이해
- 실제 검색 엔진 구현 및 활용

## 🎯 오늘 만들 것
**노래 가사 검색 엔진**: 가사 내용을 입력하면 관련 노래를 찾아주는 시스템

## 1️⃣ 이론: 텍스트 데이터와 NLP

### 텍스트 데이터의 특성
- **비정형 데이터**: 숫자와 달리 구조화되지 않은 형태
- **언어적 특성**: 문법, 문맥, 의미를 포함
- **다양성**: 언어, 문체, 길이 등이 다양함

### 자연어 처리(NLP)란?
자연어(인간이 사용하는 언어)를 컴퓨터가 이해하고 처리할 수 있도록 하는 기술

**주요 응용 분야**:
- 검색 엔진 (구글, 네이버)
- 챗봇
- 번역 서비스
- 감정 분석
- 문서 분류

## 2️⃣ 실습 준비: 라이브러리 설치 및 임포트

In [3]:
# ! pip install konlpy
# ! pip install wordcloud
# ! pip install pandas
# ! pip install numpy
# ! pip install matplotlib
# ! pip install scikit-learn
# ! pip install nltk

In [4]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ 모든 라이브러리가 성공적으로 임포트되었습니다!")

✅ 모든 라이브러리가 성공적으로 임포트되었습니다!


## 3️⃣ 데이터 탐색 및 이해

### 📊 데이터 살펴보기

In [5]:
# 데이터 로드
print("📁 데이터 로드 중...")
df = pd.read_csv('songs.csv')

print("\n📋 데이터 샘플:")
print(df.head())

print("\n📊 데이터 정보:")
df.info()

print(f"\n🎵 총 {len(df)}개의 노래 데이터가 있습니다.")

📁 데이터 로드 중...

📋 데이터 샘플:
        title                   artist   song_id  \
0      Golden                  HUNTR/X  39166708   
1    Soda Pop  KPop Demon Hunters Cast  39166705   
2      FAMOUS           ALLDAY PROJECT  39156202   
3    뛰어(JUMP)                BLACKPINK  39298775   
4  Dirty Work                    aespa  39121279   

                                              lyrics  
0  I was a ghost, I was alone, hah\n어두워진, hah, 앞길...  
1  Hey, hey\nHey, hey\nHey\nDon't want you, need ...  
2  분명 나쁜 아이는 아니어도\n또 틀에 가두면 we break it\nBum no b...  
3  I’m not that easy to tame\nYou should see me u...  
4  World domination\nI don’t gotta say it\n전엔 없던\...  

📊 데이터 정보:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    30 non-null     object
 1   artist   30 non-null     object
 2   song_id  30 non-null     int64 
 3   lyrics   30 non-null     obje

### 🔍 데이터 특징 분석

**관찰해보세요:**
- 가사 데이터의 형태 (한글/영어 혼합)
- 불필요한 단어들 (조사, 대명사 등)
- 의미 있는 단어들 (명사, 동사 등)

In [6]:
# 첫 번째 노래의 가사 자세히 보기
print("🎤 첫 번째 노래 정보:")
print(f"제목: {df.iloc[0]['title']}")
print(f"아티스트: {df.iloc[0]['artist']}")
print(f"\n가사:\n{df.iloc[0]['lyrics']}")

🎤 첫 번째 노래 정보:
제목: Golden
아티스트: HUNTR/X

가사:
I was a ghost, I was alone, hah
어두워진, hah, 앞길속에 (Ah)
Given the throne, I didn't know how to believe
I was the queen that I'm meant to be
I lived two lives, tried to play both sides
But I couldn't find my own place
Called a problem child 'cause I got too wild
But now that's how I'm getting paid, 끝없이 on stage
I'm done hidin', now I'm shinin' like I'm born to be
We dreamin' hard, we came so far, now I believe
We're goin' up, up, up, it's our moment
You know together we're glowing
Gonna be, gonna be golden
Oh, up, up, up with our voices
영원히 깨질 수 없는
Gonna be, gonna be golden
Oh, I'm done hidin' now I'm shinin' like I'm born to be
Oh, our time, no fear, no lies
That's who we're born to be
Waited so long to break these walls down
To wake up and feel like me
Put these patterns all in the past now
And finally live like the girl they all see
No more hiding, I'll be shining like I'm born to be
'Cause we are hunters, voices strong, and I know I believe
W

## 4️⃣ 이론: 텍스트 전처리

### 왜 전처리가 필요한가?
1. **노이즈 제거**: 불필요한 문자, 기호 제거
2. **불용어 제거**: 의미 없는 단어들 제거
3. **형태소 분석**: 단어의 기본 형태로 변환

### 불용어(Stopwords)란?
검색이나 분석에서 의미가 없는 단어들
- 조사: 은, 는, 이, 가, 을, 를
- 대명사: 나, 너, 그, 저
- 접속사: 그리고, 하지만
- 영어: the, a, an, and, or

In [7]:
# 불용어 리스트 정의
def load_stopwords():
    stopwords = [
        # 한글 불용어
        '은', '는', '이', '가', '을', '를', '에', '에서', '의', '로', '으로', '와', '과', '도', '만', '까지',
        '에서', '에게', '하고', '이다', '있다', '없다', '같다', '그', '저', '이', '것', '수', '등', '들',
        '때', '한', '지', '하', '오', '말', '일', '때문', '거', '게', '너무', '더', '나', '내', '걸', '이런',
        '저런', '왜', '그냥', '다시', '정도', '때문', '이제', '다시', '모두', '아니', '없이', '같이', '처럼',
        '다른', '모든', '우리', '내가', '네가', '그녀', '그들', '나의', '너의', '이것', '저것', '그것', '누구',
        '무엇', '어디', '언제', '어떻게', '왜', '몇', '얼마나', '모든', '다른', '어떤', '무슨', '아무',
        # 영어 불용어
        'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 'yourself',
        'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they',
        'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these',
        'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having',
        'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
        'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during',
        'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over',
        'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all',
        'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only',
        'own', 'same', 'so', 'than', 'too', 'very', 'can', 'will', 'just', 'should', 'now'
    ]
    return set(stopwords)

print("✅ 불용어 리스트가 준비되었습니다.")
print(f"📝 총 {len(load_stopwords())}개의 불용어가 정의되어 있습니다.")

✅ 불용어 리스트가 준비되었습니다.
📝 총 200개의 불용어가 정의되어 있습니다.


In [ ]:
# 텍스트 전처리 함수
def preprocess_text(text, remove_stopwords=True):
    import re
    
    # 1. 특수문자 제거 (한글, 영어, 공백만 남김)
    text = re.sub(r'[^가-힣a-zA-Z\s]', ' ', str(text))
    
    # 2. 불용어 제거
    if remove_stopwords:
        stopwords = load_stopwords()
        words = text.split()
        words = [word for word in words if word not in stopwords]
        text = ' '.join(words)
    
    return text.strip()

print("🔧 텍스트 전처리 함수가 준비되었습니다.")

# 전처리 전후 비교
sample_text = df.iloc[0]['lyrics'][:100]
print(f"\n📝 전처리 전: {sample_text}")
print(f"🔧 전처리 후: {preprocess_text(sample_text)}")

🔧 텍스트 전처리 함수가 준비되었습니다.

📝 전처리 전: I was a ghost, I was alone, hah
어두워진, hah, 앞길속에 (Ah)
Given the throne, I didn't know how to believe

🔧 전처리 후: I ghost I alone hah 어두워진 hah 앞길속에 Ah Given throne I didn t know believe


## 5️⃣ 이론: 형태소 분석과 명사 추출

### 형태소 분석이란?
문장을 의미 있는 최소 단위(형태소)로 분리하는 과정

**예시**: "나는 학교에 갔다"
- 나(대명사) + 는(조사) + 학교(명사) + 에(조사) + 가(동사) + 았(어미) + 다(어미)

### 왜 명사만 추출하는가?
- **의미 중심**: 명사가 가장 중요한 의미를 담고 있음
- **검색 품질**: 명사 기반 검색이 더 정확함
- **노이즈 감소**: 조사, 어미 등은 검색에 도움이 되지 않음

In [11]:
# 형태소 분석기 초기화

# 4. 형태소 분석기 초기화
tokenizer = Okt()

def get_nouns(text, min_length=1):
    # 명사만 추출하고 길이 제한 적용
    nouns = tokenizer.nouns(text)
    # 최소 길이 이상의 명사만 필터링
    nouns = [noun for noun in nouns if len(noun) >= min_length]
    return ' '.join(nouns)

print("\n4. 명사 추출 중...")
# 최소 2글자 이상의 명사만 추출
df['nouns'] = df['cleaned_lyrics'].apply(lambda x: get_nouns(x, min_length=2))


print("🔍 형태소 분석기가 준비되었습니다.")

# 명사 추출 예시
sample_text = "나는 오늘 학교에 갔다"
print(f"\n📝 원본: {sample_text}")
print(f"🔍 명사 추출: {get_nouns(sample_text)}")

# 실제 가사에서 명사 추출
sample_lyrics = df.iloc[0]['lyrics'][:200]
print(f"\n🎵 가사 샘플: {sample_lyrics}")
print(f"🔍 추출된 명사: {get_nouns(sample_lyrics)}")

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.



CalledProcessError: Command '['/usr/libexec/java_home']' returned non-zero exit status 1.

In [ ]:
# 전체 데이터에 전처리 및 명사 추출 적용
print("🔄 전체 데이터 전처리 중...")

# 불용어 제거를 포함한 텍스트 전처리
df['cleaned_lyrics'] = df['lyrics'].apply(lambda x: preprocess_text(x, remove_stopwords=True))

# 최소 2글자 이상의 명사만 추출
df['nouns'] = df['cleaned_lyrics'].apply(lambda x: get_nouns(x, min_length=2))

print("✅ 전처리가 완료되었습니다!")

# 결과 확인
print("\n📊 전처리 결과 샘플:")
for i in range(3):
    print(f"\n노래 {i+1}: {df.iloc[i]['title']}")
    print(f"원본: {df.iloc[i]['lyrics'][:100]}...")
    print(f"명사: {df.iloc[i]['nouns']}")

## 6️⃣ 이론: BOW (Bag of Words)

### BOW란?
문서를 단어의 출현 빈도로 표현하는 방법

### 특징:
- ✅ **단순함**: 구현이 쉽고 이해하기 쉬움
- ✅ **빠름**: 계산 속도가 빠름
- ❌ **순서 무시**: 단어의 순서 정보가 사라짐
- ❌ **의미 무시**: 동의어, 유사어를 다른 단어로 취급

### 예시:
문장: "나는 사랑을 한다"
- 어휘: [나, 사랑, 한다]
- BOW 벡터: [1, 1, 1] (각 단어가 1번씩 등장)

In [ ]:
# BOW 구현
def create_bow(texts):
    # 단어 사전 생성
    words = ' '.join(texts).split()
    word_counts = Counter(words)
    vocab = {word: idx for idx, word in enumerate(word_counts.keys())}
    
    # BOW 벡터 생성
    bow_vectors = []
    for text in texts:
        vector = [0] * len(vocab)
        for word in text.split():
            if word in vocab:
                vector[vocab[word]] += 1
        bow_vectors.append(vector)
    
    return bow_vectors, vocab

print("📦 BOW 생성 중...")
bow_vectors, vocab = create_bow(df['nouns'])

print(f"\n📚 어휘 크기: {len(vocab)}개 단어")
print(f"📊 문서 수: {len(bow_vectors)}개")

# 첫 번째 문서의 BOW 벡터 일부 출력
print(f"\n🔢 첫 번째 문서의 BOW 벡터 (처음 20개):")
print(bow_vectors[0][:20])

# 가장 많이 등장하는 단어들 확인
all_words = ' '.join(df['nouns']).split()
word_freq = Counter(all_words)
print(f"\n🏆 가장 많이 등장하는 단어 TOP 10:")
for word, count in word_freq.most_common(10):
    print(f"{word}: {count}회")

## 7️⃣ 이론: TF-IDF (Term Frequency-Inverse Document Frequency)

### TF-IDF란?
단어의 중요도를 계산하는 방법

### 공식:
**TF-IDF = TF × IDF**

- **TF (Term Frequency)**: 문서 내 단어 빈도
- **IDF (Inverse Document Frequency)**: 전체 문서에서 단어의 희귀성

### 장점:
- ✅ **중요도 반영**: 자주 등장하지만 희귀한 단어에 높은 점수
- ✅ **검색 품질 향상**: 의미 있는 단어를 우선시
- ✅ **노이즈 감소**: 너무 흔한 단어의 영향 감소

### 예시:
- "사랑"이 여러 문서에 등장 → 낮은 IDF → 낮은 TF-IDF
- "우주"가 특정 문서에만 등장 → 높은 IDF → 높은 TF-IDF

In [ ]:
# TF-IDF 계산
print("📊 TF-IDF 계산 중...")
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['nouns'])

print(f"\n📐 TF-IDF 행렬 크기: {tfidf_matrix.shape}")
print(f"📚 어휘 크기: {tfidf_matrix.shape[1]}개 단어")
print(f"📄 문서 수: {tfidf_matrix.shape[0]}개 문서")

In [ ]:
# 첫 번째 문서의 TF-IDF 값 분석
first_doc_tfidf = tfidf_matrix[0].toarray()[0]
feature_names = tfidf_vectorizer.get_feature_names_out()

print("🔍 첫 번째 문서의 TF-IDF 점수 (0이 아닌 값들):")
nonzero_scores = [(feature_names[i], first_doc_tfidf[i]) 
                  for i in range(len(first_doc_tfidf)) 
                  if first_doc_tfidf[i] > 0]

# 점수 순으로 정렬
nonzero_scores.sort(key=lambda x: x[1], reverse=True)

for word, score in nonzero_scores[:10]:
    print(f"{word}: {score:.4f}")

In [ ]:
# 특정 단어의 TF-IDF 점수 분석
word = "사랑"
if word in tfidf_vectorizer.vocabulary_:
    idx = tfidf_vectorizer.vocabulary_[word]
    word_tfidf_values = tfidf_matrix[:, idx].toarray().flatten()
    
    print(f"🎵 '{word}' 단어의 각 문서별 TF-IDF 점수:")
    
    # 0이 아닌 값들만 출력
    nonzero_indices = [i for i, score in enumerate(word_tfidf_values) if score > 0]
    
    for idx in nonzero_indices:
        print(f"문서 {idx+1} ({df.iloc[idx]['title']}): {word_tfidf_values[idx]:.4f}")
else:
    print(f"❌ '{word}' 단어가 사전에 없습니다.")

## 8️⃣ 시각화: 단어 구름 (Word Cloud)

### 단어 구름이란?
텍스트에서 자주 등장하는 단어를 크기로 표현한 시각화

### 장점:
- **직관적**: 한눈에 주요 키워드 파악 가능
- **시각적**: 빈도에 따른 크기 차이로 중요도 표현
- **인사이트**: 데이터의 전반적인 주제 파악

In [ ]:
# 단어 구름 생성 함수
def generate_wordcloud(text):
    wordcloud = WordCloud(
        width=800, height=400, 
        background_color='white',
        font_path='/System/Library/Fonts/AppleSDGothicNeo.ttc'  # macOS 한글 폰트
    ).generate(text)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('🎵 노래 가사 단어 구름', fontsize=16, pad=20)
    plt.show()

print("☁️ 단어 구름 생성 중...")
all_nouns = ' '.join(df['nouns'])
generate_wordcloud(all_nouns)

print("\n💡 분석 결과:")
print("- 가장 큰 단어들이 가장 자주 등장하는 키워드입니다.")
print("- 이 단어들을 기반으로 검색이 이루어집니다.")

## 9️⃣ 이론: 검색 엔진의 원리

### 검색 엔진이 작동하는 방식

1. **쿼리 전처리**: 검색어를 정제하고 형태소 분석
2. **벡터화**: 쿼리를 TF-IDF 벡터로 변환
3. **유사도 계산**: 쿼리와 각 문서 간의 코사인 유사도 계산
4. **순위 결정**: 유사도가 높은 순서로 결과 정렬

### 코사인 유사도란?
두 벡터 간의 각도를 이용한 유사도 측정 방법
- **1.0**: 완전히 동일한 벡터
- **0.0**: 완전히 다른 벡터 (직각)
- **-1.0**: 완전히 반대 방향

### 수식:
cos(θ) = (A·B) / (||A|| × ||B||)

In [ ]:
# 검색 엔진 클래스 구현
class SimpleSearchEngine:
    def __init__(self, df, tfidf_matrix, vectorizer):
        self.df = df
        self.tfidf_matrix = tfidf_matrix
        self.vectorizer = vectorizer
    
    def search(self, query, top_n=5):
        # 1. 쿼리 전처리 및 변환
        query = preprocess_text(query)
        query_nouns = get_nouns(query)
        query_vec = self.vectorizer.transform([query_nouns])
        
        # 2. 코사인 유사도 계산
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        
        # 3. 상위 N개 결과 반환
        top_indices = similarities.argsort()[-top_n:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                'title': self.df.iloc[idx]['title'],
                'artist': self.df.iloc[idx]['artist'],
                'similarity': f"{similarities[idx]:.2%}",
                'lyrics_preview': self.df.iloc[idx]['lyrics'][:100] + '...'
            })
        
        return pd.DataFrame(results)

print("🔍 검색 엔진이 준비되었습니다!")

In [ ]:
# 검색 엔진 초기화 및 테스트
print("🚀 검색 엔진 초기화 중...")
search_engine = SimpleSearchEngine(df, tfidf_matrix, tfidf_vectorizer)

# 테스트 검색
test_query = '사랑'
print(f"\n🔍 테스트 검색: '{test_query}'")
results = search_engine.search(test_query)

print("\n📋 검색 결과:")
print(results[['title', 'artist', 'similarity']])

print("\n💡 결과 해석:")
print("- 유사도가 높을수록 검색어와 더 관련성이 높습니다.")
print("- 0%는 해당 단어가 가사에 없다는 의미입니다.")

## 🔬 실험: 다양한 검색어로 테스트

### 여러 검색어로 검색해보기

In [ ]:
# 다양한 검색어로 테스트
test_queries = ['사랑', '이별', '밤', '꿈', '행복', '슬픔', '기쁨']

for query in test_queries:
    print(f"\n🔍 검색어: '{query}'")
    results = search_engine.search(query, top_n=3)
    
    if len(results) > 0:
        print("📋 상위 3개 결과:")
        for idx, row in results.iterrows():
            print(f"  {idx+1}. {row['title']} - {row['artist']} (유사도: {row['similarity']})")
    else:
        print("❌ 검색 결과가 없습니다.")

## 🎯 실습: 직접 검색해보기

### 여러분이 원하는 검색어로 테스트해보세요!

In [ ]:
# 직접 검색해보기
def interactive_search():
    while True:
        query = input("\n🔍 검색어를 입력하세요 (종료: 'quit'): ")
        
        if query.lower() == 'quit':
            print("👋 검색을 종료합니다.")
            break
        
        if query.strip() == '':
            print("❌ 검색어를 입력해주세요.")
            continue
        
        print(f"\n🔍 '{query}' 검색 결과:")
        results = search_engine.search(query)
        
        if len(results) > 0:
            for idx, row in results.iterrows():
                print(f"\n{idx+1}. {row['title']} - {row['artist']}")
                print(f"   유사도: {row['similarity']}")
                print(f"   가사 미리보기: {row['lyrics_preview']}")
        else:
            print("❌ 검색 결과가 없습니다.")

# 주석을 해제하고 실행하면 대화형 검색을 사용할 수 있습니다
# interactive_search()

## 📊 성능 분석 및 개선 방안

### 현재 시스템의 장단점

**장점:**
- ✅ 구현이 간단하고 이해하기 쉬움
- ✅ 빠른 검색 속도
- ✅ 기본적인 검색 기능 제공

**단점:**
- ❌ 문맥 이해 부족
- ❌ 동의어 처리 불가
- ❌ 검색 품질이 데이터에 의존적

### 개선 방안

1. **불용어 사전 확장**
   - 더 많은 불용어 추가
   - 도메인별 특화 불용어

2. **형태소 분석 개선**
   - 동의어 사전 구축
   - 유사어 매핑

3. **고급 기술 적용**
   - BERT 기반 임베딩
   - Word2Vec, FastText
   - 딥러닝 기반 검색

4. **사용자 경험 개선**
   - 검색 결과 하이라이트
   - 자동완성 기능
   - 검색 히스토리

## 🎓 학습 정리

### 오늘 배운 것들:

1. **텍스트 데이터의 특성**
   - 비정형 데이터의 특징
   - 자연어 처리의 필요성

2. **텍스트 전처리**
   - 불용어 제거의 중요성
   - 형태소 분석과 명사 추출

3. **벡터화 방법**
   - BOW: 단순한 단어 빈도 기반
   - TF-IDF: 중요도 기반 가중치

4. **검색 엔진 구현**
   - 코사인 유사도 계산
   - 실제 검색 시스템 구축

### 다음 단계:
- 더 큰 데이터셋으로 확장
- 고급 NLP 기술 학습 (BERT, Transformer)
- 실제 서비스에 적용

### 실무 적용 분야:
- 검색 엔진 (구글, 네이버)
- 챗봇 및 대화 시스템
- 문서 분류 및 요약
- 감정 분석
- 번역 서비스

## 🎉 수고하셨습니다!

오늘 우리는 자연어 처리의 기초부터 실제 검색 엔진 구현까지 배웠습니다.

**핵심 포인트:**
- 텍스트 데이터는 특별한 전처리가 필요합니다
- TF-IDF는 검색 품질을 크게 향상시킵니다
- 간단한 아이디어로도 유용한 시스템을 만들 수 있습니다

**질문이나 추가 학습이 필요하시면 언제든 말씀해주세요!** 🚀